# 🕵️ Validación de Integridad
Consultas que detectan inconsistencias reales encontradas en los datos: relaciones rotas, entidades incompletas, estados imposibles, valores inválidos y registros inconsistentes.

In [ ]:
# 1.1 RELACIONES ROTAS

# Pedidos cuyo customer_id no existe en customers

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    o.order_id,
    o.customer_id,
    o.current_status
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
ORDER BY o.order_id;
"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PEDIDOS CON CLIENTES INEXISTENTES ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:
# 1.2 RELACIONES ROTAS

# Items cuyo order_id no existe en orders

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    oi.order_item_id,
    oi.order_id,
    oi.product_id
FROM order_items oi
LEFT JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_id IS NULL
ORDER BY oi.order_item_id;
"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- ITEMS CON PEDIDOS INEXISTENTES ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:
# 1.3 RELACIONES ROTAS

# Pagos cuyo order_id no existe en orders

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    p.payment_id,
    p.order_id,
    p.amount,
    p.payment_status
FROM payments p
LEFT JOIN orders o ON p.order_id = o.order_id
WHERE o.order_id IS NULL
ORDER BY p.payment_id;
"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PAGOS CON PEDIDOS INEXISTENTES ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:
# 2.1 ENTIDADES INCOMPLETAS

# Pedidos sin ningún item asociado

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    o.order_id,
    o.customer_id,
    o.current_status,
    o.order_total,
    o.order_datetime
FROM orders o
LEFT JOIN order_items oi ON o.order_id = oi.order_id
WHERE oi.order_item_id IS NULL
ORDER BY o.order_datetime DESC;

"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PEDIDOS SIN NINGÚN ITEM ASOCIADO ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:
# 2.2 ENTIDADES INCOMPLETAS

# Pedidos sin ningún pago registrado

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    o.order_id,
    o.customer_id,
    o.current_status,
    o.order_total,
    o.order_datetime
FROM orders o
LEFT JOIN payments p ON o.order_id = p.order_id
WHERE p.payment_id IS NULL
ORDER BY o.order_datetime DESC;

"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PEDIDOS SIN NINGÚN PAGO REGISTRADO ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:
# 3 ESTADOS IMPOSIBLES

# Pedidos marcados como 'delivered' pero sin historial de 'shipped'

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    o.order_id,
    o.current_status,
    o.order_datetime,
    o.order_total
FROM orders o
LEFT JOIN order_status_history h 
    ON o.order_id = h.order_id 
    AND h.status = 'shipped'
WHERE o.current_status = 'delivered'
AND h.status_history_id IS NULL
ORDER BY o.order_datetime DESC;

"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PEDIDOS ENTREGADOS QUE NUNCA FUERON ENVIADOS ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:
# 4 VALORES INVÁLIDOS

# Pagos con estado 'approved' pero monto = 0

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    p.payment_id,
    p.order_id,
    p.payment_status,
    p.amount,
    p.method,
    p.payment_datetime
FROM payments p
WHERE p.payment_status = 'approved'
AND p.amount = 0
ORDER BY p.payment_datetime DESC;

"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PAGOS APROBADOS CON MONTO IGUAL A CERO ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:
# 5.1 REGISTROS INCONSISTENTES

# Pedidos cancelados con pagos aprobados

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    o.order_id,
    o.current_status,
    p.payment_id,
    p.payment_status,
    p.amount,
    p.payment_datetime
FROM orders o
JOIN payments p ON o.order_id = p.order_id
WHERE o.current_status = 'cancelled'
AND p.payment_status = 'approved'
ORDER BY p.amount DESC;
"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PEDIDOS CANCELADOS QUE TIENEN PAGOS APROBADOS ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:
# 5.2 REGISTROS INCONSISTENTES

# Clientes inactivos con pedidos activos

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    c.customer_id,
    c.full_name,
    c.is_active,
    o.order_id,
    o.current_status,
    o.order_datetime
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE c.is_active = 0
AND o.is_active = 1
ORDER BY o.order_datetime DESC;
"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PEDIDOS ACTIVOS DE CLIENTES MARCADOS COMO INACTIVOS ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:
# 5.3 REGISTROS INCONSISTENTES

# Pedidos con order_total = 0 pero con items

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    o.order_id,
    o.order_total,
    o.current_status,
    oi.order_item_id,
    oi.line_total
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_total = 0
ORDER BY o.order_id;
"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PEDIDOS CON TOTAL 0 PERO QUE CONTIENEN ITEMS ---")
for fila in resultados:
    print(fila)

conexion.close()

# 🔍 Consultas Estructurales
Consultas que permiten navegar el modelo, relacionar entidades, rastrear estados y detectar ausencias en la base de datos.

In [ ]:

# Ver todos los pedidos con el nombre del cliente

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    o.order_id,
    c.full_name,
    c.city,
    c.segment,
    o.order_datetime,
    o.current_status,
    o.order_total,
    o.channel
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_datetime DESC;
"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PEDIDOS CON DATOS DEL CLIENTE ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:

# Ver el detalle de items de cada pedido con nombre del producto

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    o.order_id,
    c.full_name,
    p.product_name,
    p.category,
    p.brand,
    oi.quantity,
    oi.unit_price,
    oi.discount_rate,
    oi.line_total
FROM order_items oi
JOIN orders o     ON oi.order_id   = o.order_id
JOIN customers c  ON o.customer_id = c.customer_id
JOIN products p   ON oi.product_id = p.product_id
ORDER BY o.order_id ASC;
"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- DETALLE DE ITEMS CON PRODUCTO Y CLIENTE ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:

# Ver el historial completo de cambios de estado de cada pedido

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    o.order_id,
    c.full_name,
    o.current_status,
    h.status          AS estado_historico,
    h.changed_at,
    h.changed_by,
    h.reason
FROM order_status_history h
JOIN orders o    ON h.order_id    = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_id ASC, h.changed_at ASC;
"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- HISTORIAL COMPLETO DE ESTADOS POR PEDIDO ---")
for fila in resultados:
    print(fila)

conexion.close()

In [ ]:

# Productos que nunca fueron incluidos en ningún pedido

import sqlite3

conexion = sqlite3.connect("database.db")
cursor = conexion.cursor()

query = """
SELECT 
    p.product_id,
    p.product_name,
    p.category,
    p.brand,
    p.unit_price,
    p.is_active
FROM products p
LEFT JOIN order_items oi ON p.product_id = oi.product_id
WHERE oi.order_item_id IS NULL
ORDER BY p.category ASC;
"""

cursor.execute(query)
resultados = cursor.fetchall()

print("--- PRODUCTOS QUE NUNCA FUERON PEDIDOS ---")
for fila in resultados:
    print(fila)

conexion.close()